# Lecture 13 — Linear Differential Equations & Schrödingerization (UnitaryLab companion)

Many scientific problems reduce to a linear ODE system $\dot{u}=Lu$ (here $L$ is a
discretized differential operator). When $L$ is anti-Hermitian the flow is unitary
and we can simulate it directly; when $L$ has a dissipative (Hermitian, negative
semidefinite) part — as for the **heat equation** $u_t = a\,u_{xx}$ — the dynamics
are *non-unitary*. **Schrödingerization** lifts such a system to a higher-dimensional
*unitary* one by introducing an auxiliary variable $p$, so a quantum computer can
simulate it with Hamiltonian-simulation machinery; the physical solution is recovered
by reading out a slice at a chosen $p$.

`ua.HeatEquationAlgorithm().run(params=None)` solves the default 1-D heat problem

$$u_t = a\,u_{xx},\quad x\in[0,L],\quad u(x,0)=\sin(2\pi x/L),\quad u(0,t)=u(L,t)=0,$$

with $a=1$, $L=17$, final time $T=1$, on $2^5=32$ interior grid points (Dirichlet
boundary). We compare its recovered profile against **two** classical references:
the analytic Fourier eigenmode and a finite-difference `expm` of the discrete
Laplacian.

Contents:
1. Setup
2. Introspect the heat-equation result dictionary
3. Classical reference 1: analytic eigenmode
4. Classical reference 2: discrete-Laplacian `expm`
5. Plot quantum vs. classical
6. Your turn

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive")
import contextlib, io
import unitarylab_algorithms as ua

@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        yield

try:
    from scipy.linalg import expm as _expm
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False
    def _expm(M):
        # M is real symmetric here (a * discrete Laplacian * T): use eigh.
        w, V = np.linalg.eigh(M)
        return (V * np.exp(w)) @ V.T

np.set_printoptions(precision=4, suppress=True)
print("scipy available:", _HAVE_SCIPY)

scipy available: True


## 2. Introspect the heat-equation result dictionary

The result carries the spatial grid (`x`), the recovered solution profile (`u`), and
grid metadata (`n_points`, `dx`). We read them out and recover the problem
parameters $a$, $L$, $T$ from the defaults.

In [2]:
with quiet():
    res = ua.HeatEquationAlgorithm().run(params=None, backend="numpy")
print("keys:", list(res.keys()))
print("status:", res["status"], "|", res["message"])

x = np.array(res["x"])
u_q = np.array(res["u"])
dx = res["grid"]["dx"]
N = res["grid"]["n_points"]

# Default problem parameters (from the algorithm's setup.json).
a, L, T = 1.0, 17.0, 1.0
print(f"grid: {N} interior points, dx = {dx:.5f}, domain [0, {L}]")
print("x[:4] =", np.round(x[:4], 4))
print("u(x,T)[:4] =", np.round(u_q[:4], 4))

keys: ['status', 'message', 'grid', 'x', 'u', 'circuit', 'plot']
status: ok | Heat equation solved
grid: 32 interior points, dx = 0.51515, domain [0, 17.0]
x[:4] = [0.5152 1.0303 1.5455 2.0606]
u(x,T)[:4] = [0.1653 0.3246 0.4722 0.6027]


## 3. Classical reference 1: analytic eigenmode

The initial condition $u_0=\sin(2\pi x/L)$ is an eigenfunction of $\partial_{xx}$ with
eigenvalue $-(2\pi/L)^2$, so the exact solution of $u_t=a\,u_{xx}$ is

$$u(x,t)=e^{-a(2\pi/L)^2 t}\,\sin(2\pi x/L).$$

The quantum-recovered profile should match this to within the Schrödingerization /
discretization error.

**Tolerance:** relative $L^2$ error $< 10^{-2}$.

In [3]:
decay = np.exp(-a * (2 * np.pi / L) ** 2 * T)
u_analytic = decay * np.sin(2 * np.pi * x / L)
rel_err_an = np.linalg.norm(u_q - u_analytic) / np.linalg.norm(u_analytic)
print("decay factor exp(-a (2pi/L)^2 T):", round(decay, 6))
print("relative L2 error (quantum vs analytic):", format(rel_err_an, ".3e"))
assert rel_err_an < 1e-2
print("Verified: recovered heat profile matches the analytic eigenmode (rel L2 < 1e-2).")

decay factor exp(-a (2pi/L)^2 T): 0.872316
relative L2 error (quantum vs analytic): 1.250e-03
Verified: recovered heat profile matches the analytic eigenmode (rel L2 < 1e-2).


## 4. Classical reference 2: discrete-Laplacian `expm`

Independently, we integrate the *same* discretization classically. The Dirichlet
Laplacian on the interior points is the tridiagonal matrix
$\dfrac{a}{\Delta x^2}\,\mathrm{tridiag}(1,-2,1)$, and $u(T)=e^{L_h T}u_0$ via a dense
matrix exponential (`scipy.linalg.expm`, with an `eigh` fallback). This is the
classical finite-difference reference that the quantum method reproduces.

**Tolerance:** relative $L^2$ error $< 10^{-2}$.

In [4]:
main = -2.0 * np.ones(N)
off = np.ones(N - 1)
Lap = (np.diag(main) + np.diag(off, 1) + np.diag(off, -1)) * (a / dx**2)
u0 = np.sin(2 * np.pi * x / L)
u_fd = _expm(Lap * T) @ u0

rel_err_fd = np.linalg.norm(u_q - u_fd) / np.linalg.norm(u_fd)
print("relative L2 error (quantum vs finite-difference expm):", format(rel_err_fd, ".3e"))
assert rel_err_fd < 1e-2
# The two classical references must also agree with each other.
assert np.linalg.norm(u_fd - u_analytic) / np.linalg.norm(u_analytic) < 5e-3
print("Verified: quantum matches finite-difference expm, and both references agree.")

relative L2 error (quantum vs finite-difference expm): 8.370e-04
Verified: quantum matches finite-difference expm, and both references agree.


## 5. Plot quantum vs. classical

We overlay the initial profile, the quantum-recovered solution at $t=T$, and the two
classical references. The mode has decayed by the factor computed above while keeping
its sinusoidal shape.

In [5]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(x, u0, "k--", lw=1, label=r"$u(x,0)=\sin(2\pi x/L)$")
ax.plot(x, u_analytic, "-", lw=2, label="analytic  $u(x,T)$")
ax.plot(x, u_fd, ".", ms=8, label="finite-difference expm")
ax.plot(x, u_q, "x", ms=6, label="quantum (Schrödingerization)")
ax.set_xlabel("x"); ax.set_ylabel("u")
ax.set_title("1-D heat equation, T = 1: quantum vs. classical")
ax.legend(fontsize=8); ax.grid(True, ls=":")
plt.tight_layout(); plt.show()
print("all three solution curves coincide within the stated tolerance.")

all three solution curves coincide within the stated tolerance.


## 6. Your turn
- The companion `ua.AdvectionEquationAlgorithm` solves $u_t + c\,u_x = 0$ (a *unitary*
  transport problem). Run it with `params=None`, introspect its `x`/`u`, and compare
  against a shifted initial profile $u_0(x-cT)$.
- Refine the grid (more spatial qubits) and watch the finite-difference reference
  converge to the analytic eigenmode as $\Delta x^2$ shrinks.
- Superpose two eigenmodes in $u_0$ and verify each decays at its own rate
  $e^{-a k^2 t}$ — the diffusion operator diagonalizes in the Fourier basis.

Lecture 14 turns to **open-system** (dissipative) dynamics and a **nonlinear** PDE,
where Schrödingerization and the Cole–Hopf transform play analogous linearizing roles.